In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print(tf.__version__)

2026-05-13 13:20:30.842572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778678431.322947      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778678431.461331      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778678432.489436      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778678432.489487      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778678432.489490      23 computation_placer.cc:177] computation placer alr

2.19.0


In [2]:
data_dir = "/kaggle/input/datasets/vuppalaadithyasairam/bone-fracture-detection-using-xrays"
train_dir = "/kaggle/input/datasets/vuppalaadithyasairam/bone-fracture-detection-using-xrays/archive (6)/train"

val_dir = "/kaggle/input/datasets/vuppalaadithyasairam/bone-fracture-detection-using-xrays/archive (6)/val"
test_dir = val_dir

- Displaying a data image

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [4]:
class_names = ['Fractured', 'Not Fractured']

In [5]:
augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

I0000 00:00:1778678476.658355      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778678476.664611      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode='binary',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 8863 files belonging to 2 classes.
Found 600 files belonging to 2 classes.
Found 600 files belonging to 2 classes.


In [7]:
train_dataset = train_dataset.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)

In [8]:
labels = np.concatenate([
    y.numpy().flatten() for x, y in train_dataset
])

labels = labels.astype(int)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = {
    0: class_weights[0],
    1: class_weights[1]
}

print(class_weights)

{0: np.float64(0.9891741071428571), 1: np.float64(1.0110654802646588)}


In [9]:
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = augmentation(inputs)
x = preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.5)(x)

x = layers.Dense(256, activation='relu')(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

model = Model(inputs, outputs)

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

In [13]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stop, reduce_lr, checkpoint],
    class_weight=class_weights
)

Epoch 1/20


I0000 00:00:1778678515.495785      69 cuda_dnn.cc:529] Loaded cuDNN version 91002


277/277 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.6866 - loss: 0.6949
Epoch 1: val_accuracy improved from -inf to 0.66500, saving model to best_model.keras
277/277 ━━━━━━━━━━━━━━━━━━━━ 92s 205ms/step - accuracy: 0.6867 - loss: 0.6945 - val_accuracy: 0.6650 - val_loss: 0.5765 - learning_rate: 0.0010
Epoch 2/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.7886 - loss: 0.4538
Epoch 2: val_accuracy improved from 0.66500 to 0.67167, saving model to best_model.keras
277/277 ━━━━━━━━━━━━━━━━━━━━ 21s 75ms/step - accuracy: 0.7886 - loss: 0.4538 - val_accuracy: 0.6717 - val_loss: 0.6022 - learning_rate: 0.0010
Epoch 3/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.8102 - loss: 0.4102
Epoch 3: val_accuracy improved from 0.67167 to 0.74833, saving model to best_model.keras
277/277 ━━━━━━━━━━━━━━━━━━━━ 21s 76ms/step - accuracy: 0.8103 - loss: 0.4102 - val_accuracy: 0.7483 - val_loss: 0.5002 - learning_rate: 0.0010
Epoch 4/20
277/277 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - 

In [14]:
base_model.trainable = True

In [15]:
for layer in base_model.layers[:100]:
    layer.trainable = False

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ModelCheckpoint("/kaggle/working/best_model.keras", save_best_only=True, monitor="val_loss")
]

In [18]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks= callbacks,
    class_weight=class_weights
)


Epoch 1/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 123s 245ms/step - accuracy: 0.6717 - loss: 0.7049 - val_accuracy: 0.8100 - val_loss: 0.3700 - learning_rate: 1.0000e-05
Epoch 2/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 146ms/step - accuracy: 0.7529 - loss: 0.5380 - val_accuracy: 0.8267 - val_loss: 0.3926 - learning_rate: 1.0000e-05
Epoch 3/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.7933 - loss: 0.4469 - val_accuracy: 0.8083 - val_loss: 0.3935 - learning_rate: 1.0000e-05
Epoch 4/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.8366 - loss: 0.3743 - val_accuracy: 0.7783 - val_loss: 0.4073 - learning_rate: 1.0000e-05
Epoch 5/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.8711 - loss: 0.3008 - val_accuracy: 0.7667 - val_loss: 0.4354 - learning_rate: 1.0000e-05
Epoch 6/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 146ms/step - accuracy: 0.8815 - loss: 0.2708 - val_accuracy: 0.7650 - val_loss: 0.4421 - learning_rate: 5.0000e-06
Epoch 7/15
277/277 ━━━━━━━━━━━━━━━━━━━━ 40s 1

In [19]:
loss, accuracy = model.evaluate(test_dataset)

print(f'Test Accuracy: {accuracy:.4f}')

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.8320 - loss: 0.3325
Test Accuracy: 0.8100


In [20]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    preds = model.predict(images)

    y_true.extend(labels.numpy())
    y_pred.extend((preds > 0.5).astype(int).flatten())

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [21]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

         0.0       0.81      0.89      0.85       360
         1.0       0.81      0.69      0.74       240

    accuracy                           0.81       600
   macro avg       0.81      0.79      0.80       600
weighted avg       0.81      0.81      0.81       600



In [22]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[321  39]
 [ 75 165]]


In [23]:
import json

class_names = ['Fractured', 'Not Fractured']

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)

print("SAVED : /kaggle/working/class_names.json")

SAVED : /kaggle/working/class_names.json
